# Eidetic Example: Common vs Specific Interfaces

This notebook demonstrates:
- Common interface (`ingest/remember/recall/forget/compact`)
- Plugin switching with stable business code
- Specific interface access via `memory.async_handle.backend`


In [ ]:
from eidetic import (
    MemoryManager,
    Document,
    MemoryEvent,
    RecallQuery,
    ForgetRequest,
    CompactRequest,
)

manager = MemoryManager()

## 1) Common Interface: business code remains stable

In [ ]:
def run_common_flow(system: str):
    memory = manager.create(system, config={"plugin_config": {"mode": "mock"}})

    memory.ingest([
        Document(content=f"{system}: unified memory API", tags=["guide", "api"]),
        Document(content=f"{system}: retrieval with one query", tags=["guide", "retrieval"]),
    ])
    memory.remember(MemoryEvent(content=f"{system}: user prefers concise style", tags=["profile"]))

    recall = memory.recall(RecallQuery(query="unified memory API", top_k=3))
    print(system, "->", [item.content for item in recall.items])

    memory.forget(ForgetRequest(tags=["profile"], hard_delete=True))
    memory.compact(CompactRequest(strategy="keep_recent", max_items=10))
    return memory

m1 = run_common_flow("letta")
m2 = run_common_flow("graphrag")

## 2) Specific Interface: plugin-native escape hatch

Use this only when you need backend-specific capabilities.
It increases coupling to a single memory system.

In [ ]:
backend = m1.async_handle.backend
print("backend type:", type(backend).__name__)
print("healthcheck:", m1.healthcheck())

# If your plugin exposes extra methods, call them behind feature checks:
# if hasattr(backend, "native_graph_stats"):
#     print(backend.native_graph_stats())

## 3) Recommended pattern

- Put product logic on the common interface.
- Isolate specific interface calls in a thin adapter module.
- Keep fallback behavior when plugin-native features are unavailable.